# Lab 8 – RNN, LSTM and Attention
COS30018 Intelligent Systems · Anas Al Azad

**Task:** every example is a sequence of T random digits and the label is the **first** digit, so a model only gets it right if it remembers step 1 until the last step. Chance = 0.10.

* **Part 1** – how the gradient shrinks over 50 steps (RNN / LSTM / GRU)
* **Part 2** – train RNN / LSTM / GRU for T = 5, 10, 20, 60 (3 seeds each) + the 128-unit experiment
* **Part 3** – self-attention for T = 10, 20, 60, where it looks, and the ablation without positional encoding
* then: results table, checkpoint answers, and one extra experiment of mine

All the code below is from the lab sheet. The only things I added are marked `# (my addition)` – saving the plots for my summary and keeping the accuracies so I can build the results table at the end.

## 3. Setup

In [ ]:
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
import os, time                                      # (my addition)
os.makedirs("figures", exist_ok=True)                # (my addition) plots for the summary go here
print("torch", torch.__version__, "| threads:", torch.get_num_threads())   # (my addition)

def make_data(n, T, seed):
    g = torch.Generator().manual_seed(seed)
    X = torch.randint(0, 10, (n, T), generator=g)   # n sequences of T digits
    return X, X[:, 0].clone()                        # label = the FIRST digit

X, y = make_data(3, 10, seed=0)
print(X)
print(y)

3 rows of 10 digits and 3 labels – each label is the first digit of its row ✔

## 4. Part 1: Vanishing gradients
The gradient at step t = how much the last hidden state changes if you nudge the input at step t.

In [ ]:
def grad_per_step(layer, T=50):
    x = torch.randn(64, T, 16, requires_grad=True)   # random input sequence
    h, _ = layer(x)                                   # h: hidden state at every step
    g, = torch.autograd.grad(h[:, -1].pow(2).sum(), x)
    return g.norm(dim=-1).mean(0).detach()            # gradient size at each input step

grad_ratio = {}                                       # (my addition)
plt.figure(figsize=(6, 3.5))
for name, Layer in [("RNN", nn.RNN), ("LSTM", nn.LSTM), ("GRU", nn.GRU)]:
    torch.manual_seed(0)
    n = grad_per_step(Layer(16, 32, batch_first=True))
    plt.semilogy(n.numpy(), label=name)
    print(name, "gradient at last step / first step = %.1e" % (n[-1] / n[0]))
    grad_ratio[name] = (n[-1] / n[0]).item()          # (my addition)
plt.xlabel("input step (0 = first)"); plt.ylabel("gradient size")
plt.title("Part 1: gradient size at each input step (T = 50)")   # (my addition)
plt.legend(); plt.tight_layout()
plt.savefig("figures/8_1_gradient_plot.png", dpi=150)            # (my addition)
plt.show()

## 5. Part 2: RNN, LSTM and GRU
Same model except for the recurrent layer; predicts the first digit from the **last** hidden state. 15 epochs, chance = 0.10.

In [ ]:
results = {}                                         # (my addition) (name, T) -> list of accuracies

class RNNNet(nn.Module):
    def __init__(self, kind, units=32):
        super().__init__()
        self.emb = nn.Embedding(10, 16)
        Layer = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
        self.rnn = Layer(16, units, batch_first=True)
        self.out = nn.Linear(units, 10)

    def forward(self, x):
        h, _ = self.rnn(self.emb(x))
        return self.out(h[:, -1])                     # predict from the last step only

def train_eval(model, T, seed, epochs=15):
    Xtr, ytr = make_data(5000, T, seed)
    Xte, yte = make_data(1000, T, seed + 1000)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 64):
            idx = perm[i:i + 64]
            opt.zero_grad()
            loss_fn(model(Xtr[idx]), ytr[idx]).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return (model(Xte).argmax(1) == yte).float().mean().item()

def run(name, make, Ts, seeds=(0, 1, 2), epochs=15):
    for T in Ts:
        accs = []
        for s in seeds:
            torch.manual_seed(s)
            accs.append(train_eval(make(T), T, s, epochs))
        print(f"{name:10s} T={T:3d} mean={np.mean(accs):.2f}      runs={np.round(accs, 2)}")
        results[(name, T)] = accs                     # (my addition)

3 models × 4 lengths × 3 seeds (the sheet says ~5 minutes):

In [ ]:
t0 = time.time()                                     # (my addition)
for kind in ["rnn", "lstm", "gru"]:
    run(kind, lambda T: RNNNet(kind), [5, 10, 20, 60])
print(f"took {time.time() - t0:.0f}s")                # (my addition)

### Experiment: change one parameter – hidden layer 32 → 128 units, T = 20 again

In [ ]:
for kind in ["rnn", "lstm", "gru"]:
    run(kind + "-128", lambda T: RNNNet(kind, units=128), [20])

## 6. Part 3: Self-attention
No recurrence – every position looks at every other position directly. Positional encoding adds a fixed sin/cos pattern so each position gets its own tag.

In [ ]:
def pos_enc(T, d):
    pos = torch.arange(T).unsqueeze(1).float()
    i = torch.arange(d).unsqueeze(0)
    angle = pos / torch.pow(10000.0, (2 * (i // 2)).float() / d)
    return torch.where(i % 2 == 0, torch.sin(angle), torch.cos(angle))

class AttnNet(nn.Module):
    def __init__(self, T, use_pos=True, d=32):
        super().__init__()
        self.use_pos = use_pos
        self.emb = nn.Embedding(10, d)
        self.register_buffer("pe", pos_enc(T, d))
        self.attn = nn.MultiheadAttention(d, 2, batch_first=True)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, d))
        self.out = nn.Linear(d, 10)

    def forward(self, x, return_w=False):
        h = self.emb(x)
        if self.use_pos:
            h = h + self.pe                           # add position information
        a, w = self.attn(h, h, h, need_weights=True, average_attn_weights=False)
        h = self.ln1(h + a)
        h = self.ln2(h + self.ff(h))
        o = self.out(h[:, -1])                        # last position makes the prediction
        return (o, w) if return_w else o

run("attn", lambda T: AttnNet(T), [10, 20, 60])

### Where does the model look?
Train one model at T = 20, average the last position's attention weights over 200 sequences, plot both heads.

In [ ]:
T = 20
torch.manual_seed(0)
model = AttnNet(T)
print("accuracy:", train_eval(model, T, seed=0))

Xs, _ = make_data(200, T, seed=5)
model.eval()
with torch.no_grad():
    _, w = model(Xs, return_w=True)                  # shape: (200, heads, T, T)
last = w[:, :, -1, :].mean(0)                        # last position, averaged over 200 sequences

fig, axes = plt.subplots(1, 2, figsize=(8, 2.8), sharey=True)
for h in range(2):
    axes[h].bar(range(T), last[h].numpy())
    axes[h].set_title(f"Head {h}"); axes[h].set_xlabel("position looked at")
axes[0].set_ylabel("attention weight")
plt.tight_layout()
plt.savefig("figures/8_2_attention_plot.png", dpi=150)          # (my addition)
plt.show()
print("weight on position 0:", [round(last[h][0].item(), 2) for h in range(2)])   # (my addition)

### Ablation: remove positional encoding

In [ ]:
run("attn_nopos", lambda T: AttnNet(T, use_pos=False), [10, 30])

## Results table (Parts 2 and 3) – mean test accuracy over 3 seeds (chance = 0.10)

In [ ]:
# (my addition) build the model x T table from everything above
import pandas as pd
order = [("rnn", "RNN"), ("lstm", "LSTM"), ("gru", "GRU"),
         ("rnn-128", "RNN (128 units)"), ("lstm-128", "LSTM (128 units)"), ("gru-128", "GRU (128 units)"),
         ("attn", "Attention"), ("attn_nopos", "Attention, no pos. enc.")]
Ts = sorted({T for (_, T) in results})
table = pd.DataFrame(index=[label for _, label in order], columns=[f"T={T}" for T in Ts], dtype=object)
for key, label in order:
    for T in Ts:
        if (key, T) in results:
            table.loc[label, f"T={T}"] = f"{np.mean(results[(key, T)]):.2f}"
table = table.fillna("–")
table.to_csv("results_table.csv")
table

In [ ]:
# (my addition) accuracy vs sequence length
plt.figure(figsize=(6, 3.6))
for key, label, style in [("rnn", "RNN", "o-"), ("lstm", "LSTM", "s-"), ("gru", "GRU", "^-"),
                          ("attn", "Attention", "D-"), ("attn_nopos", "Attention, no pos. enc.", "x--")]:
    pts = sorted((T, np.mean(a)) for (k, T), a in results.items() if k == key)
    plt.plot([p[0] for p in pts], [p[1] for p in pts], style, label=label)
plt.axhline(0.10, color="grey", ls=":", label="chance (0.10)")
plt.xlabel("sequence length T"); plt.ylabel("mean test accuracy (3 seeds)")
plt.title("How far back can each model remember?")
plt.ylim(0, 1.05); plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("figures/8_3_accuracy_vs_T.png", dpi=150)
plt.show()

In [ ]:
# (my addition) save everything for the summary / worklog
import json
json.dump({"grad_ratio": grad_ratio,
           "results": {f"{k}|{T}": v for (k, T), v in results.items()},
           "attention_weight_pos0": [round(last[h][0].item(), 4) for h in range(2)]},
          open("results.json", "w"), indent=1)
print(json.dumps(grad_ratio, indent=1))

## 7. Checkpoint questions
*(answers written after looking at my results – see the cell below)*

ANSWERS_PLACEHOLDER

## My extra experiment: is it memory or just not enough training?
At T = 20 the plain models are around chance. If I give them more epochs (40 instead of 15), do they get there eventually?

In [ ]:
# (my addition) same models, T = 20, 40 epochs instead of 15
t0 = time.time()
for kind in ["rnn", "lstm", "gru"]:
    run(kind + "-40ep", lambda T: RNNNet(kind), [20], epochs=40)
print(f"took {time.time() - t0:.0f}s")

EXTRA_PLACEHOLDER